# 1970 songs: preprocessing, exploration, features and communities

Run the cells from top to bottom. First generate `songs_1970.csv` and `genius_songs_trimmed.csv` using the existing download scripts. The analysis outputs are written to `analysis/`.

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data_exploration.py').exists():
    ROOT = next(parent for parent in [ROOT, *ROOT.parents] if (parent / 'data_exploration.py').exists())
DATA_DIR = next((folder for folder in (ROOT / 'Data', ROOT / 'data', ROOT) if folder.exists()), ROOT)
sys.path.insert(0, str(ROOT))
SONGS_PATH = DATA_DIR / 'songs_1970.csv'
RAW_LYRICS_PATH = DATA_DIR / 'genius_songs_trimmed.csv'
LYRICS_PATH = DATA_DIR / 'song_lyrics_1970.csv'
OUTPUT_PATH = ROOT / 'analysis'
OUTPUT_PATH.mkdir(exist_ok=True)
print(f'Project root: {ROOT}')
print(f'Data folder: {DATA_DIR}')
print(f'Songs available: {SONGS_PATH.exists()}')
print(f'Lyrics available: {RAW_LYRICS_PATH.exists()}')

Project root: c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject
Data folder: c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject\Data
Songs available: True
Lyrics available: True


## 1. Match and preprocess lyrics

This keeps only lyrics whose normalized title and artist occur in the MusicBrainz song table. Normalization is case-insensitive and removes bracketed annotations such as `[Remastered]`.

In [3]:
from filter_song_lyrics import filter_lyrics

if not SONGS_PATH.exists() or not RAW_LYRICS_PATH.exists():
    print('Generate songs_1970.csv and genius_songs_trimmed.csv before running this cell.')
else:
    LYRICS_PATH.parent.mkdir(exist_ok=True)
    matched = filter_lyrics(SONGS_PATH, RAW_LYRICS_PATH, LYRICS_PATH)
    print(f'Matched lyric rows: {matched}')

Matched lyric rows: 381


## 2. Load data and inspect quality

In [4]:
import pandas as pd
from data_exploration import load_songs, load_lyrics, clean_lyrics

songs = load_songs(SONGS_PATH)
lyrics = load_lyrics(LYRICS_PATH)
songs = songs.merge(
    lyrics[['song_key', 'artist_key', 'lyrics_clean']],
    on=['song_key', 'artist_key'], how='left'
)
songs['lyrics_clean'] = songs['lyrics_clean'].fillna('')
songs_without_genre = songs['genre'].str.strip().eq('').sum()
songs = songs[songs['genre'].str.strip().ne('')].copy()
display(songs.head())
display(pd.DataFrame({
    'rows': [len(songs)],
    'songs_without_genre_removed': [songs_without_genre],
    'unique_songs': [songs['id'].nunique()],
    'unique_artists': [songs['artist_key'].nunique()],
    'missing_lyrics': [(songs['lyrics_clean'] == '').sum()],
    'duplicate_ids': [songs['id'].duplicated().sum()],
}))

ImportError: DLL load failed while importing hashing: En politik for programkontrol har blokeret denne fil.

## 3. Visualize the dataset

In [ ]:
from data_exploration import write_exploration_plots

write_exploration_plots(songs, OUTPUT_PATH)
display(songs['genre'].replace('', 'Unknown').value_counts().head(15).to_frame('songs'))
display(songs['country'].replace('', 'Unknown').value_counts().head(15).to_frame('songs'))
display(songs['lyrics_clean'].str.split().str.len().describe().to_frame('words'))
print(f'Plots written to {OUTPUT_PATH}')

,songs
genre,
Unknown,783
rock,212
soul,174
pop,135
country,126
psychedelic rock,52
funk,50
progressive rock,50
jazz,48


,songs
country,
US,836
GB,316
NL,247
DE,135
FR,82
IE,78
CA,66
AU,48
JM,36


,words
count,2013.000000
mean,34.442623
std,84.038530
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,546.000000


Plots written to c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject\analysis


In [ ]:
# Check how many songs are left if we remove those with unknown genre or country
print(f"Original number of songs: {len(songs)}")
print(f"Number of songs after filtering: {len(songs_filtered)}")

Original number of songs: 2013


NameError: name 'songs_filtered' is not defined

## 4. Build features and cluster songs

Features include multi-label genres, language/country metadata, and lyric TF-IDF unigrams/bigrams. K-means is used as an initial baseline; inspect the resulting groups rather than treating the number of clusters as final.

In [ ]:
from data_exploration import build_features

build_features(songs, OUTPUT_PATH)
clusters = pd.read_csv(OUTPUT_PATH / 'song_clusters.csv') if (OUTPUT_PATH / 'song_clusters.csv').exists() else pd.DataFrame()
if not clusters.empty:
    display(clusters.groupby('cluster').size().rename('songs').to_frame())
    display(clusters.merge(songs[['id', 'genre']], on='id').sort_values('cluster').head(20))
else:
    print('Not enough usable rows/features for K-means.')

c:\Users\cheli\miniforge3\envs\Temp\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=8.
  warnings.warn(


,songs
cluster,
0,470
1,217
2,177
3,118
4,247
5,152
6,489
7,143


,id,song,artist,cluster,genre
1995,4fce69ef-957b-4234-ad84-c7b6701e7155,Take Me In Your Arms,Stern Selection,0,
1996,d6674e83-2925-4b40-b3f3-db11b5fea073,Make Love Not War,John Woolley & Just Born,0,
1997,8039bd2e-0db3-41a5-bfa4-c52d6142aed9,Ruby Baby,John Woolley & Just Born,0,
2000,e3a3a3aa-c476-421b-94a1-2d86c7b26896,Take a Ride in the Sky,Socrates Drank the Conium,0,progressive rock
2002,62cc3811-48f6-4361-a0a3-c011b663288b,Sad Memories,Stern Selection,0,
2004,8661929b-f94f-44ad-86f7-440fc9ed1fb2,Return to Italy,Eric,0,
2005,00bd88f9-9257-42a8-b29f-50c66dbe5fb6,Good Days Have Gone,Demis Roussos,0,
2012,eebb9ff5-aeda-4259-b2b7-437e193894a1,Walking 'round,Eric Spitzer-Marlyn,0,
2011,e6de875c-e8ca-498b-b4d6-411c56b552d4,Hot Pants,Salvage,0,
2010,4801f245-8dd4-4e73-865d-29a660cd161b,The Foundation of Love,Salvage,0,


## 5. Build the first community model

The graph links songs that share an artist or genre. Greedy modularity detects communities in the weighted song projection. The GEXF file can be opened in Gephi for further inspection.

In [ ]:
from data_exploration import build_communities
import matplotlib.pyplot as plt
import networkx as nx

build_communities(songs, OUTPUT_PATH)
communities = pd.read_csv(OUTPUT_PATH / 'song_communities.csv')
display(communities.groupby('community').agg(songs=('id', 'size'), mean_degree=('degree', 'mean')).sort_values('songs', ascending=False).head(20))
display(communities.merge(songs[['id', 'song', 'artist', 'genre']], on='id').sort_values(['community', 'degree'], ascending=[True, False]).head(30))

graph = nx.read_gexf(OUTPUT_PATH / 'song_similarity.gexf')
community_by_id = communities.set_index('id')['community'].to_dict()
ordered_nodes = sorted(graph.nodes, key=lambda node: community_by_id.get(str(node), -1))
positions = nx.circular_layout(ordered_nodes)
node_colors = [community_by_id.get(str(node), -1) for node in ordered_nodes]

figure, axis = plt.subplots(figsize=(12, 12))
nx.draw_networkx_edges(graph, positions, ax=axis, alpha=0.02, width=0.25, edge_color='black')
nx.draw_networkx_nodes(
    graph, positions, nodelist=ordered_nodes, node_color=node_colors,
    cmap='tab20', node_size=9, linewidths=0, ax=axis
)
axis.set_title('Songs arranged in a circle and coloured by community')
axis.axis('off')
display(figure)
plt.close(figure)

ImportError: DLL load failed while importing hashing: En politik for programkontrol har blokeret denne fil.

## 6. Save the final cleaned table

This table is the starting point for downstream graph analysis or predictive modelling.

In [ ]:
songs.to_csv(OUTPUT_PATH / 'songs_clean.csv', index=False)
print(f'Saved {len(songs)} rows to {OUTPUT_PATH / "songs_clean.csv"}')

Saved 2013 rows to c:\Users\cheli\OneDrive\Dokumenter\GitHub\SocialGraphs_FinalProject\analysis\songs_clean.csv
